# HEART DISEASE PREDICTION MODEL - ADVANCED UPGRADE
## Step 1: Enhanced Dependencies

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, StackingClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import joblib
import warnings
warnings.filterwarnings('ignore')

## Step 2: Data Loading and EDA

In [ ]:
heart_data = pd.read_csv('/content/data.csv')
print(f"Dataset shape: {heart_data.shape}")
heart_data.head()

In [ ]:
print("Missing values:")
print(heart_data.isnull().sum())
print("\nClass distribution:")
print(heart_data['target'].value_counts())
print(f"\nClass balance: {heart_data['target'].value_counts(normalize=True)}")

In [ ]:
plt.figure(figsize=(10, 6))
sns.heatmap(heart_data.corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

## Step 3: Feature Engineering and Scaling

In [ ]:
X = heart_data.drop(columns='target', axis=1)
Y = heart_data['target']

X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, stratify=Y, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Training set: {X_train_scaled.shape}")
print(f"Test set: {X_test_scaled.shape}")

## Step 4: Multiple Model Training and Comparison

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=42),
    'SVM': SVC(probability=True, random_state=42)
}

results = {}
for name, model in models.items():
    model.fit(X_train_scaled, Y_train)
    y_pred = model.predict(X_test_scaled)
    y_pred_proba = model.predict_proba(X_test_scaled)[:, 1]
    
    results[name] = {
        'model': model,
        'accuracy': accuracy_score(Y_test, y_pred),
        'precision': precision_score(Y_test, y_pred),
        'recall': recall_score(Y_test, y_pred),
        'f1': f1_score(Y_test, y_pred),
        'roc_auc': roc_auc_score(Y_test, y_pred_proba)
    }

results_df = pd.DataFrame(results).T
print("Model Comparison:")
print(results_df[['accuracy', 'precision', 'recall', 'f1', 'roc_auc']].round(4))

## Step 5: Hyperparameter Tuning (Best Model)

In [ ]:
param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [10, 20, None],
    'min_samples_split': [2, 5]
}

rf_grid = GridSearchCV(RandomForestClassifier(random_state=42), param_grid, cv=5, scoring='roc_auc', n_jobs=-1)
rf_grid.fit(X_train_scaled, Y_train)

print(f"Best parameters: {rf_grid.best_params_}")
print(f"Best cross-validation score: {rf_grid.best_score_:.4f}")

best_rf = rf_grid.best_estimator_

## Step 6: Ensemble Stacking

In [ ]:
estimators = [
    ('rf', best_rf),
    ('gb', GradientBoostingClassifier(n_estimators=100, random_state=42)),
    ('svm', SVC(probability=True, random_state=42))
]

stacking_model = StackingClassifier(
    estimators=estimators,
    final_estimator=LogisticRegression(max_iter=1000),
    cv=5
)

stacking_model.fit(X_train_scaled, Y_train)
y_pred_stack = stacking_model.predict(X_test_scaled)
y_pred_proba_stack = stacking_model.predict_proba(X_test_scaled)[:, 1]

print("Stacking Ensemble Results:")
print(f"Accuracy: {accuracy_score(Y_test, y_pred_stack):.4f}")
print(f"Precision: {precision_score(Y_test, y_pred_stack):.4f}")
print(f"Recall: {recall_score(Y_test, y_pred_stack):.4f}")
print(f"F1-Score: {f1_score(Y_test, y_pred_stack):.4f}")
print(f"ROC-AUC: {roc_auc_score(Y_test, y_pred_proba_stack):.4f}")

## Step 7: Deep Learning Model

In [ ]:
nn_model = keras.Sequential([
    layers.Dense(64, activation='relu', input_shape=(X_train_scaled.shape[1],)),
    layers.Dropout(0.3),
    layers.Dense(32, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(16, activation='relu'),
    layers.Dense(1, activation='sigmoid')
])

nn_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

history = nn_model.fit(
    X_train_scaled, Y_train,
    epochs=100,
    batch_size=32,
    validation_split=0.2,
    verbose=0
)

y_pred_nn = (nn_model.predict(X_test_scaled) > 0.5).astype(int).flatten()
y_pred_proba_nn = nn_model.predict(X_test_scaled).flatten()

print("Neural Network Results:")
print(f"Accuracy: {accuracy_score(Y_test, y_pred_nn):.4f}")
print(f"Precision: {precision_score(Y_test, y_pred_nn):.4f}")
print(f"Recall: {recall_score(Y_test, y_pred_nn):.4f}")
print(f"F1-Score: {f1_score(Y_test, y_pred_nn):.4f}")
print(f"ROC-AUC: {roc_auc_score(Y_test, y_pred_proba_nn):.4f}")

In [ ]:
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train')
plt.plot(history.history['val_accuracy'], label='Validation')
plt.title('Model Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train')
plt.plot(history.history['val_loss'], label='Validation')
plt.title('Model Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.tight_layout()
plt.show()

## Step 8: Feature Importance Analysis

In [ ]:
feature_importance = pd.DataFrame({
    'feature': X.columns,
    'importance': best_rf.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(data=feature_importance, x='importance', y='feature')
plt.title('Feature Importance (Random Forest)')
plt.tight_layout()
plt.show()

print(feature_importance)

## Step 9: Confusion Matrix Visualization

In [ ]:
cm = confusion_matrix(Y_test, y_pred_stack)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix - Stacking Ensemble')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.show()

print("\nClassification Report:")
print(classification_report(Y_test, y_pred_stack, target_names=['No Disease', 'Disease']))

## Step 10: Model Persistence

In [ ]:
joblib.dump(stacking_model, 'heart_disease_stacking_model.pkl')
joblib.dump(scaler, 'heart_disease_scaler.pkl')
nn_model.save('heart_disease_nn_model.h5')

print("Models saved successfully!")
print("- heart_disease_stacking_model.pkl")
print("- heart_disease_scaler.pkl")
print("- heart_disease_nn_model.h5")

## Step 11: Prediction Pipeline

In [ ]:
def predict_heart_disease(input_data, model_type='stacking'):
    input_array = np.asarray(input_data).reshape(1, -1)
    input_scaled = scaler.transform(input_array)
    
    if model_type == 'stacking':
        prediction = stacking_model.predict(input_scaled)[0]
        probability = stacking_model.predict_proba(input_scaled)[0]
    elif model_type == 'neural_network':
        prediction = (nn_model.predict(input_scaled, verbose=0) > 0.5).astype(int)[0][0]
        probability = nn_model.predict(input_scaled, verbose=0)[0]
    
    result = "Heart Disease" if prediction == 1 else "No Heart Disease"
    confidence = probability[1] if model_type == 'stacking' else probability[0]
    
    return result, confidence

test_input = (41, 0, 1, 130, 204, 0, 0, 172, 0, 1.4, 2, 0, 2)
result, confidence = predict_heart_disease(test_input, 'stacking')
print(f"Prediction: {result}")
print(f"Confidence: {confidence:.2%}")

test_input2 = (51, 0, 0, 130, 305, 0, 1, 142, 1, 1.2, 1, 0, 3)
result2, confidence2 = predict_heart_disease(test_input2, 'stacking')
print(f"\nPrediction: {result2}")
print(f"Confidence: {confidence2:.2%}")

## Step 12: Final Model Comparison Summary

In [ ]:
final_comparison = pd.DataFrame({
    'Model': ['Logistic Regression', 'Random Forest', 'Gradient Boosting', 'SVM', 'Stacking Ensemble', 'Neural Network'],
    'Accuracy': [
        results['Logistic Regression']['accuracy'],
        results['Random Forest']['accuracy'],
        results['Gradient Boosting']['accuracy'],
        results['SVM']['accuracy'],
        accuracy_score(Y_test, y_pred_stack),
        accuracy_score(Y_test, y_pred_nn)
    ],
    'ROC-AUC': [
        results['Logistic Regression']['roc_auc'],
        results['Random Forest']['roc_auc'],
        results['Gradient Boosting']['roc_auc'],
        results['SVM']['roc_auc'],
        roc_auc_score(Y_test, y_pred_proba_stack),
        roc_auc_score(Y_test, y_pred_proba_nn)
    ]
})

print("\n=== FINAL MODEL COMPARISON ===")
print(final_comparison.round(4))
print(f"\nBest Model: {final_comparison.loc[final_comparison['ROC-AUC'].idxmax(), 'Model']}")
print(f"Best ROC-AUC Score: {final_comparison['ROC-AUC'].max():.4f}")